# AIFM Real Estate Fund

This notebook presents a real estate risk-monitoring workflow for a simulated AIF portfolio. The fund combines direct property investments across European office, logistics, retail, and residential assets with listed REITs, cash, and FX-hedged exposures.

The analysis focuses on property and sleeve-level risk indicators, including LTV, rental stress, valuation sensitivity, occupancy, direct-property exposure, and listed REIT liquidity where relevant. Regulatory context is AIFMD-style fund governance, but the notebook focuses on the monitoring workflow and interpretation of the resulting risk metrics.

> **Output gallery:** Tables and plots generated by this notebook are saved in the [`figs/aifm_real_estate`](../../figs/aifm_real_estate) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

> For supporting methodology notes, see [Real Estate Notes](../../docs/notebooks_notes/real_estate.md).

> **Review note:** This notebook is being aligned to a closed-ended mixed real estate AIF workflow. Direct-property risk should be interpreted through property valuation, LTV, rental income, occupancy, tenant risk, refinancing, and stress testing. Listed REITs, cash, and FX hedges may still use position-style analytics where relevant. VaR, ADV-based liquidity, and redemption-stress sections are retained only where they support the listed or liquid sleeve and should be reviewed in a later real estate-specific refactor.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# to initialize db if no tables exist abd to create session factory
from src.setup_db import run as setup_db
setup_db()
from src.data.database import get_engine
ENGINE = get_engine()

# initialize mock bloomberg API
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
BBG = Bloomberg()

# helper functions for risk computations and nice output formatting
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml

### Fund Example in this Notebook

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

<small>

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID        = 'AIFM_RealEstate'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

The RMP is loaded as `rmp` and passed to the relevant risk functions. This keeps fund-specific parameters outside the calculation code.

In [ ]:
# read in RMP parameters for the fund
from src.data.reference_data import load_rmp
rmp = load_rmp(FUND_ID)

### Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.

In [ ]:
# fixed valuation date for all computations in this notebook
from src.config import VALUATION_DATE
VALUATION_DATE

Portfolio positions, fund characteristics, counterparty records, reference data, and market data are retrieved from the SQLite data layer. Market data is enriched through the simulated Bloomberg workflow before being passed to the risk analytics modules.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# enrich fund postion data w/ info from Bloomberg
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# display the enriched risk dataframe first 2 rows
risk_df.head(2)

From this point onward, the notebook uses helper functions to render summary tables and HTML views. Data aggregation, calculations, and reporting logic remain inside the package modules, so the notebook stays focused on review and interpretation.

In [ ]:
# query from SQLdb
from src.data.database import query_positions
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV, export_id="03")

In [ ]:
phtml.display_top_positions(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

In [ ]:
phtml.display_asset_class_breakdown(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")  # Also computes NAV internally

---

### Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration and used throughout the notebook for measurement, monitoring, and limit checks.

In [ ]:
# from src.config import VALUATION_DATE, QUARTER
# from src.risk.reg_constants import CONFIDENCE, HORIZON, NOTICE, GROSS_LIMIT

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from src.ui.plot_style import (
#     C,  FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
#     apply_ax_style, section_title, fig_header, fig_footer,
#     callout_box, threshold_vline, threshold_hline, breach_fill,
#     pct_color, rag_color, util_color, liq_color, table_cell, sup_title,
# )
# from src.ui.nb_utils import save_fig, save_table, styled_table, save_table_png

# import warnings
# warnings.filterwarnings('ignore')

# from src.setup_db import run as setup_db
# setup_db()

# from src.data.database import get_engine, query_positions, query_nav_history
# from src.data.enrichment import get_risk_ready_df
# from src.data.mock_bloomberg import MockBloomberg as Bloomberg
# from src.risk.leverage_config import INSTRUMENT_SOURCE
# from src.risk.risk_utils import (
#     exception_report, full_backtest_report,
#     stress_equity, stress_rates, stress_credit,
#     stress_fx, stress_combined, stress_historical,
#     stress_property, stress_rental, stress_ltv,
#     days_to_liquidate, liquidity_buckets, redemption_stress, investor_concentration,
#     liquidity_adjusted_var,
# )
# from src.risk.esg_utils import build_esg_df, esg_portfolio_summary, ESG_FIELDS, ESG_THRESHOLD_LOW

# from sqlalchemy.orm import Session
# import sqlalchemy as sa

# FUND_ID        = 'AIFM_RealEstate'
# QUARTER        = '2026-03-31'   # latest appraiser quarter
# ENGINE         = get_engine()

# BBG        = Bloomberg()


In [ ]:
# Display fund overview banner — fund identity and regulatory classification
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
)

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
)

---

## 1. Load and Validate Real Estate Exposures

Positions are retrieved from the SQLite data layer. The Real Estate fund holds two types of assets with fundamentally different data sources and risk characteristics:

- **Direct properties**: quarterly appraisal valuation, no Bloomberg ticker, fund admin data only (LTV, rental yield, vacancy rate, property type)
- **Listed REITs**: daily market prices, Bloomberg enrichment (beta, ADV)

`get_risk_ready_df` routes each position to the correct enrichment source automatically based on the `is_direct_property` flag.

In [ ]:


positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
risk_df   = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)
NAV       = risk_df['market_value_eur'].sum()

direct    = risk_df[risk_df['is_direct_property'] == True]
listed    = risk_df[risk_df['is_direct_property'] == False]

print(f"Fund              : {FUND_ID}")
print(f"Valuation date    : {VALUATION_DATE}")
print(f"Total positions   : {len(positions)}")
print(f"Direct properties : {len(direct)}")
print(f"Listed REITs/FX   : {len(listed)}")
print(f"NAV (EUR)         : {NAV:,.0f}")
print(f"Direct prop (EUR) : {direct['market_value_eur'].sum():,.0f}")
print(f"Listed (EUR)      : {listed['market_value_eur'].sum():,.0f}")

In [ ]:
# Asset class breakdown
breakdown = risk_df.groupby('asset_class').agg(
    market_value_eur=('market_value_eur', 'sum'),
    n_positions=('isin', 'count'),
).sort_values('market_value_eur', ascending=False)

breakdown['weight_pct'] = breakdown['market_value_eur'] / NAV * 100

print(f"{'Asset Class':<20} {'MV (EUR)':>15} {'Weight':>8} {'# Pos':>6}")
print('-' * 52)
for ac, row in breakdown.iterrows():
    print(f"{ac:<20} {row['market_value_eur']:>15,.0f} {row['weight_pct']:>7.1f}% {row['n_positions']:>6}")
print('-' * 52)
print(f"{'NAV':<20} {NAV:>15,.0f} {'100.0%':>8}")



In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
colors = [ACCENT2 if v < 0 else ACCENT for v in breakdown['weight_pct']]
bars = ax.barh(breakdown.index, breakdown['weight_pct'],
               color=colors, height=0.45, alpha=0.85)
ax.axvline(0, color=C['dim'], lw=0.8)
ax.set_xlabel('Weight (% NAV)', fontsize=9)
section_title(ax, 'Asset Class Breakdown')
ax.grid(True, axis='x', alpha=0.15, linestyle='--')
for bar, val in zip(bars, breakdown['weight_pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()


# save_fig(fig, FUND_ID, '06. value bridge by company')
# sup_title(fig, 'Return Attribution — Value Bridge by Company', fontsize=13)    
# section_title(ax, f"{row['company_name']}{suffix}", fontsize=10)


> **Review note:** Direct-property VaR is not the main risk lens for this closed-ended AIF. The parametric VaR assumption is an approximation and should be reviewed in a later refactor. Listed REIT VaR may remain relevant for the liquid sleeve.

---

## 2. VaR and Expected Shortfall

Direct properties are valued quarterly by independent appraisers. The absence of
daily pricing makes historical simulation VaR inapplicable to this portion of the
portfolio. AIFMD does not mandate VaR for real estate AIFs; regulatory risk
monitoring focuses on leverage, liquidity mismatch, and stress testing.

VaR is computed as follows:
- **Listed REITs**: historical simulation on daily NAV returns
- **Direct properties**: parametric VaR using an assumed annual volatility
  calibrated to MSCI Real Estate index data, disclosed in the RMP as an approximation
- **Combined**: liquid and illiquid VaR aggregated assuming independence

---

## 3. Direct Property Analysis

Key metrics for direct property positions: LTV, rental yield, and vacancy rate.
Sourced from the fund administrator quarterly valuation report.

- **LTV**: debt as % of property value. Covenant typically at 60-65%.
- **Rental yield**: annual rent / property value. Gross, before vacancy.
- **Vacancy rate**: % of lettable space not generating income.
- **Effective yield**: rental yield × (1 − vacancy rate). Actual income yield.

In [ ]:
prop_df = direct[direct['asset_class'] == 'Real Estate'].copy()
prop_df['effective_yield'] = prop_df['rental_yield_pct'] * (1 - prop_df['vacancy_rate_pct'] / 100)

summary = prop_df[[
    'instrument_name', 'country', 'market_value_eur',
    'ltv_pct', 'rental_yield_pct', 'vacancy_rate_pct',
    'effective_yield', 'property_type', 'valuation_date'
]].copy()

summary['weight_pct'] = summary['market_value_eur'] / NAV * 100

print(f"{'Property':<35} {'MV (EUR)':>12} {'Wgt':>6} {'LTV':>6} {'Yield':>7} {'Vacancy':>8} {'Eff.Yield':>10}")
print('-' * 92)
for _, row in summary.iterrows():
    print(f"{row['instrument_name']:<35} {row['market_value_eur']:>12,.0f} "
          f"{row['weight_pct']:>5.1f}% {row['ltv_pct']:>5.1f}% "
          f"{row['rental_yield_pct']:>6.1f}% {row['vacancy_rate_pct']:>7.1f}% "
          f"{row['effective_yield']:>9.1f}%")
print('-' * 92)

wav_ltv    = (summary['ltv_pct'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
wav_yield  = (summary['rental_yield_pct'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
wav_vac    = (summary['vacancy_rate_pct'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
wav_eff    = (summary['effective_yield'] * summary['market_value_eur']).sum() / summary['market_value_eur'].sum()
print(f"{'Weighted average':<35} {summary['market_value_eur'].sum():>12,.0f} "
      f"{'':>6} {wav_ltv:>5.1f}% {wav_yield:>6.1f}% {wav_vac:>7.1f}% {wav_eff:>9.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Direct Property Key Metrics', color=ACCENT, fontsize=11)

names = [n.split(' ')[0] + ' ' + n.split(' ')[1] for n in summary['instrument_name']]

for ax, col, title, limit in zip(
    axes,
    ['ltv_pct', 'rental_yield_pct', 'vacancy_rate_pct'],
    ['LTV (%)', 'Rental Yield (%)', 'Vacancy Rate (%)'],
    [60, None, None]
):
    colors = [ACCENT2 if (limit and v > limit) else ACCENT
              for v in summary[col]]
    bars = ax.barh(names, summary[col], color=colors,
                   height=0.45, alpha=0.85)
    if limit:
        ax.axvline(limit, color=ACCENT2, lw=1.2,
                   linestyle='--', label=f'Limit {limit}%')
        ax.legend(fontsize=7)
    ax.set_title(title, fontsize=9, color=ACCENT)
    ax.grid(True, axis='x', alpha=0.15, linestyle='--')
    ax.tick_params(labelsize=8, length=0)
    for bar, val in zip(bars, summary[col]):
        ax.text(bar.get_width() + 0.3,
                bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---

## 4. Leverage Monitoring

The notebook reports fund-level leverage using gross and commitment-style measures and compares the results with fund-level monitoring limits.

For this real estate AIF, leverage relates primarily to property-level debt relative to property values (LTV), and fund-level leverage relative to NAV. Gross and commitment-style measures are used where relevant for reporting inputs.

The leverage output supports Annex IV-style reporting inputs and separates financial borrowing from portfolio exposure.

In [ ]:
# MRS-23: Leverage computation - Gross and Commitment method

# ----------------------------------------------------------------
# Gross method: sum of absolute exposures / NAV
# ----------------------------------------------------------------
risk_df['abs_exposure'] = risk_df['market_value_eur'].abs()

deriv_rows     = risk_df[risk_df['asset_class'] == 'Derivative'].copy()
deriv_notional = 0.0
for _, row in deriv_rows.iterrows():
    ticker        = 'SPXW 260619P05500 Index'
    bbg_data      = BBG.bdp(ticker, ['DELTA', 'OPT_UNDL_PX', 'CONTRACT_SIZE'])
    delta         = abs(bbg_data.loc[ticker, 'DELTA'])
    undl_px       = bbg_data.loc[ticker, 'OPT_UNDL_PX']
    contract_size = bbg_data.loc[ticker, 'CONTRACT_SIZE']
    quantity      = abs(row['quantity'])
    fx_rate       = 0.89
    deriv_notional += delta * quantity * contract_size * undl_px * fx_rate

risk_df['gross_exposure'] = risk_df.apply(
    lambda r: deriv_notional if r['asset_class'] == 'Derivative'
    else (0.0 if r['asset_class'] == 'Cash'
    else r['abs_exposure']),
    axis=1
)

gross_leverage = risk_df['gross_exposure'].sum() / NAV

# ----------------------------------------------------------------
# Commitment method
# ----------------------------------------------------------------
long_eq  = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] > 0)]['market_value_eur'].sum()
short_eq = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] < 0)]['market_value_eur'].sum()
net_eq   = abs(long_eq + short_eq)
bonds    = risk_df[risk_df['asset_class'].isin(['Bond','Loan','CLO'])]['market_value_eur'].abs().sum()
fx       = risk_df[risk_df['asset_class'] == 'FX']['market_value_eur'].abs().sum()

commitment_exposure = net_eq + bonds + fx + deriv_notional
commitment_leverage = commitment_exposure / NAV

# ----------------------------------------------------------------
# Summary table
# ----------------------------------------------------------------
all_classes = sorted(risk_df['asset_class'].unique())

leverage_summary = pd.DataFrame({
    'Gross (EUR)'        : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()
                            for ac in all_classes],
    'Gross (x NAV)'      : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()/NAV
                            for ac in all_classes],
    'Commitment (EUR)'   : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional if ac == 'Derivative' else 0)
                            for ac in all_classes],
    'Commitment (x NAV)' : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()/NAV
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional/NAV if ac == 'Derivative' else 0)
                            for ac in all_classes],
}, index=all_classes)

leverage_summary['Gross (EUR)']        = leverage_summary['Gross (EUR)'].map('{:,.0f}'.format)
leverage_summary['Gross (x NAV)']      = leverage_summary['Gross (x NAV)'].map('{:.2f}x'.format)
leverage_summary['Commitment (EUR)']   = leverage_summary['Commitment (EUR)'].map('{:,.0f}'.format)
leverage_summary['Commitment (x NAV)'] = leverage_summary['Commitment (x NAV)'].map('{:.2f}x'.format)

print(f"{'Asset Class':<15} {'Gross (EUR)':>15} {'Gross':>8} {'Commit (EUR)':>15} {'Commit':>8}")
print('-' * 65)
for ac in all_classes:
    row = leverage_summary.loc[ac]
    print(f"{ac:<15} {row['Gross (EUR)']:>15} {row['Gross (x NAV)']:>8} "
          f"{row['Commitment (EUR)']:>15} {row['Commitment (x NAV)']:>8}")
print('-' * 65)
print(f"{'Total':<15} {risk_df['gross_exposure'].sum():>15,.0f} {gross_leverage:>7.2f}x "
      f"{commitment_exposure:>15,.0f} {commitment_leverage:>7.2f}x")

GROSS_LIMIT = 3.0
status      = 'OK' if gross_leverage <= GROSS_LIMIT else 'BREACH'
print(f"\nGross leverage limit : {GROSS_LIMIT:.0f}x")
print(f"Current gross        : {gross_leverage:.2f}x")
print(f"Status               : {status}")

In [ ]:
# AIFMD II granular leverage breakdown
granular = risk_df.groupby(['asset_class', 'sub_asset_class']).agg(
    gross_eur=('gross_exposure', 'sum'),
    n_positions=('isin', 'count')
).reset_index()
granular['gross_x_nav'] = granular['gross_eur'] / NAV
granular['source']      = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[0], axis=1)
granular['listed_otc']  = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[1], axis=1)

granular = granular[granular['gross_eur'] > 0].sort_values('gross_eur', ascending=False)

# listed vs OTC summary
total_gross = granular['gross_eur'].sum()
summary_lot = granular.groupby('listed_otc')['gross_eur'].sum().reset_index()
summary_lot['x_nav']        = summary_lot['gross_eur'] / NAV
summary_lot['pct_leverage'] = summary_lot['gross_eur'] / total_gross * 100
summary_lot['gross_eur']    = summary_lot['gross_eur'].map('{:,.0f}'.format)
summary_lot['x_nav']        = summary_lot['x_nav'].map('{:.2f}x'.format)
summary_lot['pct_leverage'] = summary_lot['pct_leverage'].map('{:.1f}%'.format)
summary_lot.index.name      = None
summary_lot.columns         = ['Category', 'Gross (EUR)', 'x NAV', '% Leverage']
summary_lot.set_index('Category', inplace=True)

header = f"{'':12} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_lot.iterrows():
    print(f"{idx:<12} {row['Gross (EUR)']:>15} {row['x NAV']:>8} {row['% Leverage']:>12}")
print('-' * len(header))
print()

summary_src = granular.groupby('source')['gross_eur'].sum().reset_index()
summary_src['x_nav']        = summary_src['gross_eur'] / NAV
summary_src['pct_leverage'] = summary_src['gross_eur'] / total_gross * 100
summary_src['gross_eur']    = summary_src['gross_eur'].map('{:,.0f}'.format)
summary_src['x_nav']        = summary_src['x_nav'].map('{:.2f}x'.format)
summary_src['pct_leverage'] = summary_src['pct_leverage'].map('{:.1f}%'.format)
summary_src.set_index('source', inplace=True)
summary_src.index.name      = None

header = f"{'':20} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_src.iterrows():
    print(f"{idx:<20} {row['gross_eur']:>15} {row['x_nav']:>8} {row['pct_leverage']:>12}")
print('-' * len(header))
print()

# granular table
granular['pct_leverage'] = (granular['gross_eur'] / total_gross * 100).map('{:.1f}%'.format)
granular['gross_eur']    = granular['gross_eur'].map('{:,.0f}'.format)
granular['gross_x_nav']  = granular['gross_x_nav'].map('{:.2f}x'.format)
granular.set_index(['source', 'asset_class', 'sub_asset_class'], inplace=True)
granular

---

## 5. Stress Testing

The notebook applies property value, rental stress, LTV covenant, rate, and historical stress scenarios to the real estate portfolio.

```text
Stress P&L = sensitivity × shock × market value
```

For listed REITs and liquid exposures, stress P&L uses first-order sensitivities. For direct properties, stress may involve direct revaluation based on the property value or LTV scenario.


### Scenario assumptions
Here are the assumptions used in our liquidity tests.

| Scenario | Parameter | Value | 
|----------|-----------|-------|
| Property value stress | Office | -20% | 
| Property value stress | Logistics | -20% | 
| Property value stress | Retail | -25% | 
| Property value stress | Residential | -15% | 
| Rental stress | Vacancy rate increase | +10pp | 
| Rental stress | Rental yield compression | -50bps | 
| Rate shock | Parallel shift | +200bps | 
| LTV covenant breach | Threshold | 75% | 

Historical scenario shocks are defined in `HISTORICAL_SCENARIOS` in `src/risk_utils.py`, and are printed below.

In [ ]:
# MRS-26: Annex VI stress scenarios - real estate
from src.risk.risk_utils import HISTORICAL_SCENARIOS, stress_property, stress_rental, stress_ltv

# scenario table
rows = []
for key, p in HISTORICAL_SCENARIOS.items():
    rows.append({
        'Scenario'    : p['name'],
        'Equity'      : f"{p['delta_equity']*100:.0f}%",
        'Rates (bps)' : f"{p['delta_y']*10000:.0f}",
        'Credit (bps)': f"+{p['delta_spread']*10000:.0f}",
        'USD'         : f"{p['fx_shocks'].get('USD', 0)*100:+.0f}%",
        'GBP'         : f"{p['fx_shocks'].get('GBP', 0)*100:+.0f}%",
    })
pd.DataFrame(rows).set_index('Scenario')

In [ ]:
prop = stress_property(risk_df, delta_value_by_type={
    'Office': -0.20, 'Logistics': -0.20, 'Retail': -0.25, 'Residential': -0.15
})
rent = stress_rental(risk_df, delta_vacancy=0.10, delta_yield=-0.005)
rt   = stress_rates(risk_df, delta_y=0.02)
hist = {s: stress_historical(risk_df, s) for s in HISTORICAL_SCENARIOS}

# LTV covenant breach - separate output
ltv = stress_ltv(risk_df, delta_property_value=-0.25) # assuming a severe property value shock of -25%
print(f"LTV covenant breaches: {ltv['n_breaches']}")
if ltv['n_breaches'] > 0:
    print(pd.DataFrame(ltv['breaching_properties']).to_string())
else:
    print("No LTV covenant breaches under current portfolio.")
print()

rows = [
    {'Scenario': 'Property Value Stress',   'P&L (EUR)': prop['stressed_pnl_eur'], '% NAV': prop['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Rental Stress',           'P&L (EUR)': rent['stressed_pnl_eur'], '% NAV': rent['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Rate Shock +200bps',      'P&L (EUR)': rt['stressed_pnl_eur'],   '% NAV': rt['stressed_pnl_eur']/NAV*100},
] + [
    {'Scenario': v['scenario'], 'P&L (EUR)': v['stressed_pnl_eur'], '% NAV': v['stressed_pnl_eur']/NAV*100}
    for v in hist.values()
]

summary_raw = pd.DataFrame(rows).set_index('Scenario')
worst_idx   = summary_raw['% NAV'].idxmin()

summary_raw['P&L (EUR)'] = summary_raw['P&L (EUR)'].map('{:,.0f}'.format)
summary_raw['% NAV']     = summary_raw['% NAV'].map('{:.2f}%'.format)

summary_raw.style.apply(lambda x: [
    'background-color: #7f1d1d; color: white' if i == worst_idx else ''
    for i in x.index], axis=0)

> **Note on rate shock**: `stress_rates` applies duration sensitivity to bonds
> and REITs only. For direct properties the relevant transmission mechanism is
> cap rate expansion (higher rates compress property valuations), which is not
> captured here. In practice a 200bps rate shock would also reduce direct
> property values by approximately 10-15% through cap rate widening.
> This is partially captured in the property value stress scenario above.

> **Review note:** Liquidity profile will be refactored to distinguish direct properties, listed REITs, cash, and FX hedges. Direct properties should be treated as illiquid rather than ADV-driven positions. Listed REITs and cash may still use position-style liquidity logic where relevant.

---
## 6. Liquidity Profile

ESMA guidelines (ESMA34-39-897) require AIFMs to categorise portfolio assets
by time to liquidation under normal market conditions. Results are reported
to the CSSF via Annex IV and used to assess redemption coverage.

Liquidity buckets (ESMA standard):

| Bucket | Time to liquidate |
|--------|------------------|
| 1      | 1 day            |
| 2      | 2-7 days         |
| 3      | 8-30 days        |
| 4      | 31-90 days       |
| 5      | 91-365 days      |
| 6      | > 1 year         |

ESMA buckets are roughly: day, week, month, quarter, year, or above.

Days to liquidate per position:

$$\text{days}_i = \frac{|MV_i|}{ADV_i \times \text{pct\_adv}}$$

* **ADV** (Average Daily Volume): average contracts/shares traded per day over
a 20-day window, sourced from Bloomberg. 
* **pct_adv = 25%** is an internal
RMP parameter representing the maximum fraction of ADV tradeable per day
without significant market impact. 
* Direct properties and instruments with
zero ADV are classified as illiquid (> 1 year).
* Cash and money market instruments are classified as 1 day by definition,
as they require no liquidation process.

<!-- Duplicate liquidity-profile markdown removed from displayed narrative. Cell retained to preserve notebook structure during markdown-only cleanup. -->

In [ ]:
# MRS-30: Liquidity profile
from src.config import LIQUIDITY_BUCKET_ORDER
from src.risk.risk_utils import days_to_liquidate, liquidity_buckets, redemption_stress

risk_df_liq = days_to_liquidate(risk_df, pct_adv=0.25)
risk_df_liq = liquidity_buckets(risk_df_liq)

bucket_summary = risk_df_liq.groupby('liquidity_bucket').agg(
    market_value_eur=('market_value_eur', 'sum'),
    abs_exposure=('market_value_eur', lambda x: x.abs().sum()),
    n_positions=('isin', 'count')
).reset_index()
bucket_summary['pct_nav_net'] = bucket_summary['market_value_eur'] / NAV * 100
bucket_summary['pct_nav_abs'] = bucket_summary['abs_exposure'] / NAV * 100

bucket_full = bucket_summary.set_index('liquidity_bucket').reindex(LIQUIDITY_BUCKET_ORDER).fillna(0).reset_index()

# table
print(f"{'Bucket':<15} {'Abs Exposure (EUR)':>20} {'% NAV':>8} {'# Pos':>6}")
print('-' * 55)
for _, row in bucket_full.iterrows():
    if row['abs_exposure'] > 0:
        print(f"{row['liquidity_bucket']:<15} {row['abs_exposure']:>20,.0f} "
              f"{row['pct_nav_abs']:>7.1f}% {row['n_positions']:>6.0f}")
    else:
        print(f"{row['liquidity_bucket']:<15} {'—':>20} {'—':>8} {'—':>6}")
print('-' * 55)
total_abs = risk_df_liq['market_value_eur'].abs().sum()
print(f"{'Total':<15} {total_abs:>20,.0f} {total_abs/NAV*100:>7.1f}%")


# chart
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(bucket_full['liquidity_bucket'],
              bucket_full['pct_nav_abs'],
              color=ACCENT, alpha=0.85, width=0.5)
ax.axhline(0, color=C['dim'], lw=0.8)
ax.set_ylabel('Absolute Exposure (% NAV)', fontsize=9)
ax.set_title('Liquidity Profile — Absolute Exposure by Bucket',
             color=ACCENT, fontsize=11, pad=12)
for bar, val in zip(bars, bucket_full['pct_nav_abs']):
    if val > 2:
        ax.text(bar.get_x() + bar.get_width()/2, val/2,
                f'{val:.1f}%', ha='center', va='center',
                fontsize=10, color='white', fontweight='bold')
plt.tight_layout()
plt.show()


> **Review note:** Redemption stress assumes an open-ended redemption model. For a closed-ended fund, this section should later be replaced or complemented by liquidity needs, debt service, capex, refinancing stress, and cash-reserve analysis.

---
### 6.2 Redemption Stress — MRS-31

AIFMD Art. 16 and ESMA/2020/1498 require AIFMs to assess whether the
fund can meet redemptions under normal and stressed conditions. Assets
liquidatable within the contractual notice period (buckets `'1 day'` and
`'2-7 days'`) are compared against the redemption amount. A shortfall
triggers a gate or side-pocket recommendation to the risk committee.

| Scenario | Redemption | Notice | Rationale |
| --- | --- | --- | --- |
| Normal | 10% NAV | 5 days | Routine investor exit |
| Large | 25% NAV | 5 days | Large single redemption |
| Stress | 50% NAV | 5 days | Co-ordinated stress exit |
| Largest investor | Fund-specific | 5 days | Concentration stress |

In [ ]:
# MRS-31: Redemption stress — AIFM Real Estate
NOTICE = 5   # contractual notice period (days)
_SCENARIOS = [
    (0.10, 'Normal    (10% NAV)'),
    (0.25, 'Large     (25% NAV)'),
    (0.50, 'Stress    (50% NAV)'),
]

print(f'Fund: AIFM Real Estate  |  NAV: EUR {NAV:,.0f}  |  Notice: {NOTICE} days')
print()
print(f"{'':22} {'Redemption (M)':>14} {'Liquid (M)':>12} {'Gap (M)':>12} {'Coverage':>10} Action")
print('\u2500' * 95)

_redstress = {}
for _pct, _label in _SCENARIOS:
    _r = redemption_stress(risk_df_liq, NAV, redemption_pct=_pct, notice_days=NOTICE)
    _redstress[_pct] = _r
    _gap = f"+{_r['liquidity_gap_eur']/1e6:.1f}M" if _r['liquidity_gap_eur'] >= 0 else f"{_r['liquidity_gap_eur']/1e6:.1f}M"
    print(f"{_label:<22} {_r['redemption_amount_eur']/1e6:>13.1f}M {_r['liquid_assets_eur']/1e6:>11.1f}M "
          f"{_gap:>12} {_r['coverage_ratio']:>9.2f}x  {_r['recommendation']}")
print('\u2500' * 95)
print('Largest-investor stress: see Section 6.3')

> **Review note:** Investor concentration in a closed-ended context should focus on capital-call and investor-base risk rather than redemption pressure. This section is retained but marked for refactor.

### 6.3 Investor Concentration — MRS-32

**ESMA thresholds** (ESMA/2020/1498, Annex VI):
- Single investor **> 20% of NAV** → concentration risk flag
- Top 3 investors **> 50% of NAV** → high-concentration flag

A concentrated investor base amplifies redemption risk: one large exit
can force asset sales that affect all remaining investors. The largest-investor
scenario below connects MRS-31 and MRS-32 — it uses the investor register to
derive the fourth redemption stress scenario.

In [ ]:
# MRS-32: Investor concentration — AIFM Real Estate
_investors = pd.DataFrame([
    {'investor_id': 'RE001', 'investor_name': 'Dutch Pension Fund', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.200}, 
    {'investor_id': 'RE002', 'investor_name': 'French Insurance Co', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.160}, 
    {'investor_id': 'RE003', 'investor_name': 'Belgian Pension Alliance', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.120}, 
    {'investor_id': 'RE004', 'investor_name': 'German Insurer', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.070}, 
    {'investor_id': 'RE005', 'investor_name': 'Nordic Pension', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.070}, 
    {'investor_id': 'RE006', 'investor_name': 'Austrian Pension Fund', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.060}, 
    {'investor_id': 'RE007', 'investor_name': 'Swiss RE Co', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.060}, 
    {'investor_id': 'RE008', 'investor_name': 'Family Office A', 'investor_type': 'Family Office', 'aum_eur': NAV * 0.055}, 
    {'investor_id': 'RE009', 'investor_name': 'Endowment B', 'investor_type': 'Endowment', 'aum_eur': NAV * 0.055}, 
    {'investor_id': 'RE010', 'investor_name': 'Asset Manager C', 'investor_type': 'Asset Manager', 'aum_eur': NAV * 0.050}, 
    {'investor_id': 'RE011', 'investor_name': 'Institutional Inv D', 'investor_type': 'Asset Manager', 'aum_eur': NAV * 0.050}, 
    {'investor_id': 'RE012', 'investor_name': 'Private Bank E', 'investor_type': 'Private Bank', 'aum_eur': NAV * 0.050}, 
])

_conc = investor_concentration(_investors, NAV, threshold=0.20)
_top  = _conc['by_investor']
_type = _investors.set_index('investor_id')['investor_type']

print(f'Investor Concentration — AIFM Real Estate  |  NAV: EUR {NAV:,.0f}')
print('ESMA threshold: 20% single / 50% top-3\n')
print(f"{'':4} {'Investor':<30} {'Type':<18} {'AUM (EUR)':>14} {'% NAV':>8}")
print('\u2500' * 80)
for _rank, (_, _row) in enumerate(_top.iterrows(), 1):
    _t = _type.get(_row['investor_id'], '')
    print(f"{_rank:<4} {_row['investor_name']:<30} {_t:<18} {_row['aum_eur']:>14,.0f} {_row['pct_nav']*100:>7.1f}%")
print('\u2500' * 80)

_flag_s = '\u26a0 ESMA flag'       if _conc['concentration_flag'] else '\u2713 OK'
_flag_3 = '\u26a0 High conc.'      if _conc['high_concentration'] else '\u2713 OK'
print(f"\nLargest investor : {_conc['largest_investor_pct']*100:.1f}% NAV  {_flag_s}")
print(f"Top 3 investors  : {_conc['top3_pct']*100:.1f}% NAV  {_flag_3}")

# Largest-investor redemption stress (4th scenario)
_r4   = redemption_stress(risk_df_liq, NAV, redemption_pct=_conc['largest_investor_pct'], notice_days=5)
_gap4 = f"+{_r4['liquidity_gap_eur']/1e6:.1f}M" if _r4['liquidity_gap_eur'] >= 0 else f"{_r4['liquidity_gap_eur']/1e6:.1f}M"
print(f"\nLargest-investor stress ({_conc['largest_investor_pct']*100:.1f}% NAV, 5-day notice):")
print(f"  Redemption : EUR {_r4['redemption_amount_eur']:,.0f}")
print(f"  Liquid     : EUR {_r4['liquid_assets_eur']:,.0f}")
print(f"  Gap        : {_gap4}  |  Coverage: {_r4['coverage_ratio']:.2f}x")
print(f"  Action     : {_r4['recommendation']}")

print('\nMonitoring recommendation:')
if _conc['high_concentration']:
    print('  \u2014 Enhanced monitoring: top-3 investors represent significant co-ordinated exit risk')
    print('  \u2014 Maintain liquidity buffer >= largest investor AUM')
if _conc['concentration_flag']:
    print(f'  \u2014 Gate-trigger review: largest investor at {_conc["largest_investor_pct"]*100:.1f}% NAV')
if not _conc['concentration_flag'] and not _conc['high_concentration']:
    print('  \u2014 No immediate action. Continue quarterly investor concentration monitoring.')

### 6.4 Tenant Default and Concentration Stress

For a real estate fund, the primary counterparty income risk is tenant default or vacancy. A major tenant loss causes immediate rental income shortfall and potential asset value impairment.

The notebook models the largest single tenant defaulting on its lease, resulting in rental income loss under the selected stress assumptions.

> Tenant register is simulated. A production system would derive these figures from executed lease agreements in the asset management system.

In [ ]:
# MRS-35: Counterparty stress — AIFM Real Estate
# Simulated commercial tenant register (annual contracted rent)
_tenants_re = pd.DataFrame([
    {'tenant': 'Carrefour SA',      'property': 'Paris Office Complex',  'annual_rent_eur': 3_200_000, 'lease_years_left': 7},
    {'tenant': 'Deutsche Post DHL', 'property': 'Hamburg Logistics Hub', 'annual_rent_eur': 2_800_000, 'lease_years_left': 5},
    {'tenant': 'ING Bank NV',       'property': 'Amsterdam Mixed Use',   'annual_rent_eur': 2_100_000, 'lease_years_left': 4},
    {'tenant': 'H&M Group',         'property': 'Lyon Industrial Park',  'annual_rent_eur': 1_600_000, 'lease_years_left': 6},
    {'tenant': 'Thales Group',      'property': 'Brussels Office',       'annual_rent_eur': 1_200_000, 'lease_years_left': 3},
])
_tenants_re['income_loss_eur'] = _tenants_re['annual_rent_eur']  # 1-year void, no recovery
_tenants_re['loss_pct_nav']    = _tenants_re['income_loss_eur'] / NAV

_worst_tenant   = _tenants_re.loc[_tenants_re['income_loss_eur'].idxmax()]
_tenant_loss    = _worst_tenant['income_loss_eur']
_tenant_pct     = _worst_tenant['loss_pct_nav']

print(f"Counterparty Stress — AIFM Real Estate  |  NAV: EUR {NAV:,.0f}")
print(f"1-year rental income loss (largest tenant defaults, full void assumed):\n")
print(f"{'Tenant':<22} {'Property':<26} {'Annual Rent':>13} {'Lease Left':>11} {'% NAV':>8}")
print('─' * 83)
for _, r in _tenants_re.iterrows():
    print(f"{r['tenant']:<22} {r['property']:<26} {r['annual_rent_eur']:>12,.0f} "
          f"{r['lease_years_left']:>8}y  {r['loss_pct_nav']*100:>7.2f}%")
print('─' * 83)
print(f"\nWorst-case: {_worst_tenant['tenant']} ({_worst_tenant['property']}) defaults")
print(f"  Income loss: EUR {_tenant_loss:,.0f}  ({_tenant_pct*100:.2f}% of NAV)")
print(f"  Note: income loss does not reduce NAV directly but impairs yield and asset value.")
print(f"  Implied NAV impact (income capitalised at 5% yield): EUR {_tenant_loss/0.05:,.0f}  "
      f"({_tenant_loss/0.05/NAV*100:.1f}% of NAV)")


> **Review note:** Current combined stress assumes 25% NAV redemption demand, which does not fit a closed-ended fund. Future version should combine property value decline, tenant default, refinancing stress, and cash-flow pressure.

### 6.5 Combined Stress (Market + Liquidity) — MRS-35

For real estate the relevant **market shock** is a uniform property value decline (−20% across all asset types). Combined with a **25% NAV redemption demand**, this mirrors the 2008/2022 scenario where falling asset prices coincided with forced seller pressure. The fund holds illiquid direct properties — liquid resources are limited to cash; a gap would require gating or asset disposal.


In [ ]:
# MRS-35: Combined stress — property -20% AND 25% redemption simultaneously
# prop is already in scope from Section 4 stress cell (property -20% scenario)
_comb_mkt_eur_re = prop['stressed_pnl_eur']
_comb_nav_st_re  = NAV + _comb_mkt_eur_re

# Under property stress, liquid assets (cash only for RE) are unchanged
# but the redemption demand is still 25% of pre-stress NAV
_base_red_re         = _redstress[0.25]
_comb_liquid_st_re   = _base_red_re['liquid_assets_eur']  # cash unaffected by property shock
_comb_redeem_eur_re  = NAV * 0.25
_comb_gap_st_re      = _comb_liquid_st_re - _comb_redeem_eur_re
_comb_cov_st_re      = _comb_liquid_st_re / _comb_redeem_eur_re if _comb_redeem_eur_re > 0 else float('inf')
_comb_action_re      = 'Can meet redemption' if _comb_gap_st_re >= 0 else 'Gate / property disposal required'

print(f"Combined Stress — AIFM Real Estate  |  Property −20% + 25% Redemption")
print(f"Baseline NAV: EUR {NAV/1e6:,.1f}M\n")
print(f"  Market shock (property −20% uniform):")
print(f"    Portfolio P&L  : EUR {_comb_mkt_eur_re/1e6:,.1f}M  ({_comb_mkt_eur_re/NAV*100:.1f}% NAV)")
print(f"    Stressed NAV   : EUR {_comb_nav_st_re/1e6:,.1f}M")
print()
print(f"  Liquidity impact (25% redemption, cash buffer only):")
print(f"    Redemption     : EUR {_comb_redeem_eur_re/1e6:,.1f}M  (25% pre-stress NAV)")
print(f"    Liquid assets  : EUR {_comb_liquid_st_re/1e6:,.1f}M  (cash, unaffected by property shock)")
print(f"    Liquidity gap  : EUR {_comb_gap_st_re/1e6:,.1f}M  |  Coverage: {_comb_cov_st_re:.2f}x")
print(f"    Action         : {_comb_action_re}")
print()
_total_stress_re = _comb_mkt_eur_re - max(0.0, -_comb_gap_st_re)
_total_pct_re    = _total_stress_re / NAV * 100
print(f"  Total combined impact on NAV: EUR {_total_stress_re/1e6:,.1f}M  ({_total_pct_re:.1f}% of NAV)")
print(f"  Regulatory note: ESMA/2020/1498 §48 — combined stress is a mandatory Annex VI scenario")


## 7. ESG Risk Indicators

The notebook computes portfolio-level ESG indicators using NAV-weighted averages and the fund's internal ESG threshold.

Metrics monitored include weighted average ESG score, percentage of NAV below the internal threshold, controversy flags, and carbon intensity.

ESG scores for liquid instruments are sourced from Bloomberg. Illiquid instruments without a Bloomberg ticker use fund administrator ESG data.

ESG scores should be treated as sustainability-risk inputs unless explicitly mapped to regulatory concepts such as SFDR Article 6, 8, or 9.

In [ ]:
esg_df  = build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
summary = esg_portfolio_summary(esg_df, NAV)

In [ ]:
def style_esg(row):
    styles = [''] * len(row)
    idx = esg_df.columns.tolist()
    if row.get('controversy_flag') == True:
        return ['background-color: #7f1d1d; color: white'] * len(row)
    elif pd.notna(row.get('esg_score')) and row.get('esg_score') < ESG_THRESHOLD_LOW:
        styles[idx.index('esg_score')] = 'background-color: #7f1d1d; color: white'
    return styles

esg_df.style.apply(style_esg, axis=1).format({
    'market_value_eur' : '{:,.0f}',
    'weight_pct'       : '{:.2f}%',
    'esg_score'        : '{:.0f}',
    'env_score'        : '{:.0f}',
    'soc_score'        : '{:.0f}',
    'gov_score'        : '{:.0f}',
    'carbon_intensity' : '{:.1f}',
    'esg_exposure_eur' : '{:,.0f}',
}, na_rep='—')

In [ ]:
print(f"ESG PORTFOLIO SUMMARY")
print('-' * 45)
print(f"{'Weighted avg ESG score':<30} {summary['wav_esg']:.1f}/100")
print(f"{'Weighted avg ENV score':<30} {summary['wav_env']:.1f}/100")
print(f"{'Weighted avg SOC score':<30} {summary['wav_soc']:.1f}/100")
print(f"{'Weighted avg GOV score':<30} {summary['wav_gov']:.1f}/100")
print(f"{'Weighted avg carbon intensity':<30} {summary['wav_carbon']:.1f} tCO2/EURm")
print(f"{'% exposure below ESG threshold':<30} {summary['pct_low_esg']:.1f}%")
print(f"{'% exposure with controversy':<30} {summary['pct_controversy']:.1f}%")
print()
if len(summary['controversies']) > 0:
    print("Controversy flags:")
    for _, row in summary['controversies'].iterrows():
        print(f"  {row['instrument_name']:<35} ESG: {row['esg_score']:.0f}")

In [ ]:
esg_scored = esg_df[esg_df['esg_score'].notna()].copy()
total_scored_mv = esg_scored['esg_exposure_eur'].sum()
ac_esg = esg_scored.groupby('asset_class').agg(
    wav_esg=('esg_score', lambda x: (x * esg_scored.loc[x.index, 'esg_exposure_eur']).sum() /
             esg_scored.loc[x.index, 'esg_exposure_eur'].sum()),
    exposure=('esg_exposure_eur', 'sum')
).reset_index()
ac_esg['pct_total'] = ac_esg['exposure'] / total_scored_mv * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
fig.suptitle('ESG Profile by Asset Class', color=ACCENT, fontsize=11)

colors = [C['muted'], C['dim'], C['border'], C['border'], C['text'], C['text']]
left = 0
for i, (_, row) in enumerate(ac_esg.iterrows()):
    axes[0].barh(0, row['pct_total'], left=left,
                 color=colors[i % len(colors)], alpha=0.85, height=0.4)
    if row['pct_total'] > 3:
        axes[0].text(left + row['pct_total']/2, 0,
                     f"{row['asset_class']}\n{row['pct_total']:.1f}%",
                     ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    left += row['pct_total']

axes[0].set_xlim(0, 100)
axes[0].set_yticks([])
axes[0].set_xlabel('% of ESG-scored exposure', fontsize=9)
axes[0].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
axes[0].tick_params(labelsize=9, length=0)

bars = axes[1].barh(ac_esg['asset_class'], ac_esg['wav_esg'],
                    color=[ACCENT2 if v < 50 else ACCENT3 if v < 70 else ACCENT
                           for v in ac_esg['wav_esg']],
                    height=0.4, alpha=0.85)
axes[1].axvline(ESG_THRESHOLD_LOW, color=ACCENT2, lw=1, linestyle='--',
                label=f'Low ESG threshold ({ESG_THRESHOLD_LOW})')
axes[1].axvline(70, color=ACCENT3, lw=1, linestyle='--', label='Good ESG threshold (70)')
axes[1].set_xlim(0, 100)
axes[1].set_xlabel('Weighted avg ESG score', fontsize=9)
axes[1].spines[['top', 'right', 'left', 'bottom']].set_visible(False)
axes[1].grid(True, axis='x', alpha=0.15, linestyle='--')
axes[1].tick_params(labelsize=9, length=0)
axes[1].legend(fontsize=7)
for bar, val in zip(bars, ac_esg['wav_esg']):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---

## 8. Annex IV Reporting Context

This section summarises real estate indicators used as Annex IV-style reporting inputs: fund-level leverage, property-level LTV, liquidity profile, investor concentration, tenant concentration, property value stress, and ESG indicators.

For this closed-ended real estate AIF, the main reporting lens is property valuation, leverage, LTV covenant monitoring, and direct-property exposure. Listed REITs are monitored for liquid-market exposure and liquidity. Cash is monitored as a liquidity buffer, and FX hedges are monitored separately where relevant.

In [ ]:
from src.reporting.annex_iv import build_annex_iv, export_annex_iv_excel

ANNEX_IV_QUARTER = '2026-03-31'

annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV — {FUND_ID}   Reporting period: Q1 2026 ({ANNEX_IV_QUARTER})")
print(f"NAV: EUR {annex_iv['_nav']/1e6:,.1f}M")

print("\n=== Section 1 — Fund Identification ===")
display(annex_iv['identification'])


In [ ]:
print("\n=== Section 2.1 — Asset Class Breakdown ===")
display(annex_iv['asset_class_breakdown'])

print("\n=== Section 2.2 — Geographic Breakdown ===")
display(annex_iv['geography_breakdown'])

print("\n=== Section 2.4 — Top 5 Positions ===")
display(annex_iv['top5_positions'])

print("\n=== Section 3 — Risk Measures ===")
display(annex_iv['risk_measures'])

print("\n=== Section 4 — Leverage ===")
display(annex_iv['leverage_detail'])

print("\n=== Section 5 — Liquidity (direct property dominates '>365 days' bucket) ===")
liq = annex_iv['liquidity_buckets']
cols = [c for c in ['bucket', 'nav_eur', 'nav_pct', 'cumulative_pct'] if c in liq.columns]
display(liq[cols])
display(annex_iv['liquidity_terms'])
